In [ ]:
import pathlib

import altair as alt
import geopandas as gpd
import laspy
import logging
import numpy as np
import open3d as o3d
import pyproj
import shapely 

import pitchmark
import pitchmark.osm
import pitchmark.lidar

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
logging.getLogger()

In [ ]:
logging.warning("abc")
logging.info("def")

In [ ]:
map_path = pathlib.Path().cwd() / "OpenStreetMap" / "augusta_national" / "map.osm"
handler = pitchmark.osm.GolfHandler()
handler.apply_file(map_path)
fc = handler.feature_collection
augusta_national = pitchmark.Course.from_featurecollection(fc)

In [ ]:
geoseries = augusta_national.gdf.geometry
geoseries

In [ ]:
azalea = augusta_national.holes[12]
azalea

In [ ]:
redbud = augusta_national.holes[15]
redbud

In [ ]:
folder = pathlib.Path().cwd() / "USGS_LIDAR"
files = [
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1284n1253.laz",
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1284n1254.laz",
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1285n1253.laz",
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1285n1254.laz",
]
paths = [folder / file for file in files]
paths

In [ ]:
# las_classes = [
#     pitchmark.lidar.las.LasClassification.unclassified,
#     pitchmark.lidar.las.LasClassification.ground,
# ]
# pitchmark.lidar.las.clip_to_geoseries(
#     paths,
#     geoseries,
#     classification_filter=las_classes,
#     to_file=folder / "augusta_national_unclassified_or_ground.laz",
# )
ground = laspy.read(folder / "augusta_national_ground.laz")

In [ ]:
augusta_national.populate_hole_meshes(folder / "augusta_national_ground.laz")

In [ ]:
tee = list(augusta_national.transformer_to_local.itransform(azalea.path.coords))[0]

In [ ]:
azalea.mesh["yards_east"] = (azalea.mesh["x"] - tee[0])
azalea.mesh["yards_north"] = (azalea.mesh["y"] - tee[1]) 
azalea.mesh["yards_from_tee"] = np.sqrt(
    azalea.mesh["yards_east"]** 2 + azalea.mesh["yards_north"] ** 2
)
azalea.mesh

In [ ]:
azalea.mesh

In [ ]:
layout = pitchmark.plotting.chart_course(azalea.gdf)

In [ ]:
alt.Chart(azalea.mesh.iloc[::50]).mark_circle().encode(
    longitude="x",
    latitude="y",
    color="z",
    tooltip=["yards_east", "yards_north", "yards_from_tee", "z"],
).project(
    type="identity",
    reflectY=True
)

In [ ]:
azalea.mesh[azalea.mesh["grade"] <= 0.1].iloc[::20]

In [ ]:
azalea.mesh["slope_heading"] = np.degrees(
    np.arctan2(azalea.mesh["normal_x"], azalea.mesh["normal_y"])
)
azalea.mesh["slope_heading"] = np.where(
    azalea.mesh["slope_heading"] < 0,
    azalea.mesh["slope_heading"] + 360,
    azalea.mesh["slope_heading"]
)
azalea.mesh


In [ ]:
green_or_short_grass = azalea.gdf[
        (azalea.gdf.ground_cover == "green")
        | (azalea.gdf.ground_cover == "short_grass")
    ].unary_union
shapely.prepare(green_or_short_grass)
selected_mesh = azalea.mesh[azalea.mesh.within(green_or_short_grass)]

In [ ]:
selector = alt.selection_single(on="mouseover", nearest=True)
inclines = (
    alt.Chart(selected_mesh.iloc[::20])
    .mark_point(
        shape="wedge",
        filled=True,
        color="lightgray",
    )
    .encode(
        longitude="x",
        latitude="y",
        color=alt.condition(selector, alt.value("black"), alt.value("lightgray")),
        angle="slope_heading",
        size=alt.Size("grade", scale=alt.Scale(domain=[0, 0.1])),
        tooltip=["yards_east", "yards_north", "yards_from_tee", "z", "slope_heading", "grade"],
    )
    .add_selection(selector)
    .project(
        type="identity",
        reflectY=True,
    )
    .properties(
        width=800,
        height=800,
    )
)


In [ ]:
(layout + inclines)

In [ ]:
redbud = augusta_national.holes[15]
redbud

In [ ]:
redbud_flag = augusta_national.transformer_to_local.transform(*redbud.path.coords[-1])
redbud_flag

In [ ]:
redbud_green = redbud.gdf[
    redbud.gdf.contains(shapely.Point(redbud_flag))
    & (redbud.gdf["course_area"] == "putting_green")
].unary_union
redbud_green_surround = redbud_green.buffer(10.0)
shapely.prepare(redbud_green_surround)
redbud_green_surround


In [ ]:
surround_features = pitchmark.plotting.chart_course(redbud.gdf.clip(redbud_green_surround))

In [ ]:
redbud.mesh["slope_heading"] = np.degrees(
    np.arctan2(redbud.mesh["normal_x"], redbud.mesh["normal_y"])
)
redbud.mesh["slope_heading"] = np.where(
    redbud.mesh["slope_heading"] < 0,
    redbud.mesh["slope_heading"] + 360,
    redbud.mesh["slope_heading"]
)
redbud.mesh

In [ ]:
selected_mesh = redbud.mesh[redbud.mesh.within(redbud_green)]
selector = alt.selection_single(on="mouseover", nearest=True)
inclines = (
    alt.Chart(selected_mesh)
    .mark_point(
        shape="wedge",
        filled=True,
        color="lightgray",
    )
    .encode(
        longitude="x",
        latitude="y",
        color=alt.condition(
            selector,
            alt.value("red"),
            alt.Color("grade:Q", scale=alt.Scale(domain=[0, 0.12], scheme="greys")),
        ),
        angle="slope_heading",
        size="grade",
        tooltip=["z", "slope_heading", "grade"],
    )
    .add_selection(selector)
    .project(
        type="identity",
        reflectY=True,
    )
    .properties(
        width=800,
        height=800,
    )
)


In [ ]:
surround_features + inclines

In [ ]:
surround_features